# 阶段 9：性能测试

测四件事：

1. **帧率**：用 `FRAME_CNT` 在固定时间内的增量算，比软件计时准。
2. **axi-lite 读写延迟**：一次寄存器读要多久（决定调度轮询能多快）。
3. **切换响应延迟**：从写 `MODE_SEL` 到画面改用新模式，需要几个帧。
4. **资源占用**：这个要回 Vivado 看 `Report Utilization`（本 notebook 只提醒看哪里）。

In [ ]:
import os
import sys
import time

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)
vp.start_capture(vdma0, mode)
regs.wait_new_frame()
print("VERSION=0x%08X 阶段 %d" % (regs.version(), regs.version_stage()))

## 1) 帧率

1280x720@30 的期望值是 **30**（OV5640 在这个分辨率下的标称帧率，不是 60）。
2026-10-04 在阶段 8 的 60 秒日志里实测 **27.4 fps**（1642 帧 / 60.0 s），
所以 27~30 都算正常；只有掉到 25 以下，才先查采集通道和显示器，再怀疑 IP。

In [ ]:
def measure_fps(seconds=3.0):
    f0 = regs.frame_cnt()
    t0 = time.time()
    time.sleep(seconds)
    f1 = regs.frame_cnt()
    t1 = time.time()
    frames = f1 - f0
    dt = t1 - t0
    return frames / dt, frames, dt

fps, frames, dt = measure_fps(3.0)
print("FRAME_CNT: %.0f 帧 / %.2f s  ->  FPS = %.2f" % (frames, dt, fps))
if fps < 25:
    print("!! 帧率偏低（正常 27~30）。先确认采集通道确实在跑、显示器接着，")
    print("   再看 axis_video_proc 是否引入了反压（本设计是直通的，正常不该有）。")

## 2) AXI-Lite 读写延迟

这个数决定调度循环最快能多久轮询一次。一般应该在 1~3 微秒量级。

In [ ]:
def measure_axi_latency(n=2000):
    t0 = time.perf_counter()
    for _ in range(n):
        regs.rd(0x1C)
    t1 = time.perf_counter()
    return (t1 - t0) / n * 1e6   # 微秒

us = measure_axi_latency()
print("单次寄存器读: %.3f 微秒" % us)

## 3) 模式切换响应：写完之后几个帧内生效

因为 `mode_active` 只在 tuser 才更新，所以**最多 1 帧**延迟。
这里用“写完之后等多少帧，`VBLANK_CNT` 已经跨过一次”来量化。

In [ ]:
def measure_switch_latency(repeats=5):
    lat = []
    for i in range(repeats):
        regs.set_mode(i % 3)
        f0 = regs.frame_cnt()
        regs.wait_frame_boundary()          # 新模式一定在这之后生效
        lat.append(regs.frame_cnt() - f0)
    regs.set_mode(0)                        # 测完恢复模式 0，别影响后面的演示
    return lat

lat = measure_switch_latency()
print("每次切换跨越的帧数:", lat, " -> 最大", max(lat))
print("换算成时间: %.1f ms (30fps 时 1 帧 ≈ 33.3ms)" % (max(lat) * 1000.0 / max(fps, 1e-6)))

## 4) 资源占用（回 Vivado 看）

在 Vivado 里打开实现后的工程（或看 `impl_1/*.rpt`），找 **Report Utilization**，
验收目标是方案书写的：LUT < 50%、BRAM < 60%、DSP < 30%。

本工程的自定义部分非常小：

* `axis_video_proc`：几个加法器、移位器、比较器，没有 BRAM；
* `mlp_engine`：3×8 + 8×3 个乘加，全部展开，占用几个 DSP。

真正吃资源的是原来的 `v_demosaic` 和两个 `axi_vdma`，所以总量应该远低于上限。

## 结论

把上面几个数抄进答辩材料即可。若帧率明显低于 30：

1. 先确认显示器接好（`v_axi4s_vid_out` 需要外部 HDMI 提供时序参考）；
2. 再看 `insert_*.tcl` 之后的 BD 里，`axis_video_proc` 是否被插到了别的位置；
3. 最后才怀疑 IP 内部反压（本设计的握手是完全直通的，正常不该有）。